# 1. Carga de Datos XLSX a Pandas (para el análisis previo)

Con el propósito de entender los datos antes de realizar la estructura de la base de datos, primero realizaremos un análisis breve de los datos. Para ello, vamos a juntar todas las páginas del fichero en un único **Pandas DataFrame**, al que luego podremos realizar operaciones fácilmente.

El algoritmo itera sobre las páginas del DataFrame, añadiendo dos columnas nuevas:

- **`Date`**: Almacena la fecha de la muestra. La obtenemos del nombre de cada página con la función `normalize_dates()`.
- **`Group`**: Almacenará la familia a la que pertenece el compuesto, aunque de momento lo dejamos en `None`.

Tras esto, se juntan todos los DataFrames (páginas) en uno y se guarda en un archivo **XLSX** con el nombre original más el sufijo `_mod`.

In [1]:
# Imports

import pandas as pd
import os
from datetime import datetime
import openpyxl

In [3]:
# Función para poner todas las fechas en el mismo formato: 'YYYY-MM-DD'
def normalize_dates(date_list):
    output_list = []
    dt = None
    year = None
    month = None
    for date in date_list:
        try:
            # Algunas de las fechas tienen espacios, los eliminamos
            date = date.replace(" ", "")
            
            #El año siempre viene defnido por los 4 ultimos caracteres y el mes por los 2 anteriores al año
            # Algunos de los dias vienen no definidos excatemente (ej: 17_19022020), nos quedamos con el último dia del rango por comodidad
            year = date[-4:]
            month = date[-6:-4]
            day = date[-8:-6]
            
            # Creamos un datetime y lo formateamos a YYYY-MM-DD (nos vendrá bien este formato para insertar en la base de datos)
            dt = datetime(int(year), int(month), int(day))
            output_list.append(dt.strftime("%Y-%m-%d"))
        except Exception as e:
            print(f"\nError: {e}")
            print(f"Nombre de página incorrecto: {date_list[0]}, por favor corrija el nombre al formato DDMMYYYY en el archivo y pruebe de nuevo")
        
    return output_list

In [4]:
# HAY NOMBRES DE PÁGINA CON UN FORMATO INCORRECTO, ESTO GENERA ERRORES AL INTENTAR SACAR LA FECHA DEL NOMBRE

excels_dir = "../Datos Excel/LecturasIncluidas"

for excel_name in os.listdir(excels_dir):
    excel_full_dir = os.path.join(excels_dir, str(excel_name))
    excel = pd.ExcelFile(excel_full_dir)
    
    sheet_names = excel.sheet_names
    
    for names in sheet_names:
        if len(names) > 8 and " " not in names and "_" not in names:
            print(f"El excel {excel_name} tiene la pagina con nombre incorrecto: {names}")

In [4]:
# CARGAMOS LOS DATOS DE TODOS LOS EXCELS (MENOS OTROS_PUNTOS) EN UN PANDAS DATAFRAME

# Carpeta con todos los excels menos 'otros puntos'
excels_dir = "../Datos Excel/LecturasIncluidas"
excel_counter = 0
df_list = []
for excel_name in os.listdir(excels_dir):
    # Obtenemos la ubicación del excel
    excel_full_dir = os.path.join(excels_dir, str(excel_name))
    
    # Obtenemos el excel
    excel = pd.ExcelFile(excel_full_dir)
    
    # Contamos cuantas paginas tiene el xlsx
    number_sheets = len(excel.sheet_names)
    
    print(f"\n ({excel_counter+1}/{len(os.listdir(excels_dir))}) Nombre {excel_name} NumPag: {number_sheets}")
    excel_counter+=1
    
    # Iteramos por todas las paginas del xlsx para sacar todos los dataframes en una lista
    for i in range(0,number_sheets):
        
        # Si la hoja no tiene los datos que queremos se pasa a la siguiente iteración
        if len(excel.sheet_names[i]) < 8:
            continue
        
        print(f"\r\t\t Pag {i+1}", end="")
        # Cargamos la pagina en un dataframe
        # skiprows hace que no carguemos las 3 primeras filas de la pagina en el dataframe (en esas filas solo está la fecha)
        # usecols hace que solo leamos las 7 primeras columnas, ya que hay la sheet 18062018 del fichero 2575 tiene una columna extra que se lee y se añade erroneamente como columna
        df_sheet = pd.read_excel(excel_full_dir, sheet_name=i, skiprows=range(3), usecols=range(7))
        
        if str(excel.sheet_names[i]) == "020142019":
            sheet_date = normalize_dates(["02042019"])[0]
        else:
            sheet_date = normalize_dates([excel.sheet_names[i]])[0]
        
        # Añadimos la columna de fecha al datafranme y ponemos todos sus valores a la fecha obtenida anteriormente
        df_sheet.insert(7, "Date", "")
        df_sheet['Date'] = sheet_date
        
        # Añadimos la columna grupo con todos sus valores NaN por el momento
        df_sheet.insert(8, "Group", "")
        df_sheet['Group'] = pd.NA
        
        # Añadimos el dataframe modificado a la lista
        df_list.append(df_sheet)

# Unificamos todos los dataframes en uno, ignore_index hace que se cree un nuevo indice para cada fila, haciendolos únicos
df_total = pd.concat(df_list, ignore_index=True)
df_total.to_excel("../Pandas Dataframe/full_dataframe.xlsx", index=False)
df_total


 (1/19) Nombre 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 22
		 Pag 22
 (2/19) Nombre 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx NumPag: 22
		 Pag 22
 (3/19) Nombre 2575_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 22
		 Pag 22
 (4/19) Nombre 2599_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 17
		 Pag 3
 (5/19) Nombre 2602_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 19
		 Pag 18
 (6/19) Nombre 2635_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 19
		 Pag 19
 (7/19) Nombre 2640_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 22
		 Pag 22
 (8/19) Nombre 2641_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 22
		 Pag 22
 (9/19) Nombre 2642_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 20
		 Pag 20
 (10/19) Nombre 2643_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 22
		 Pag 22
 (11/19) Nombre 2645_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx NumPag: 13
		 Pag 13
 (12/19

,Component RT,Library RT,Compound Name,Match Factor,Formula,CAS#,Sample Name,Date,Group
0,22.446411,NaN,Octacosane,97.344876,C28H58,630-02-4,2554_070318_FS_SV,2018-03-17,NaN
1,8.457277,NaN,"Benzenesulfonamide, N-butyl-",97.342626,C10H15NO2S,3622-84-2,2554_070318_FS_SV,2018-03-17,NaN
2,6.200755,NaN,Dodecanoic acid,97.034088,C12H24O2,143-07-7,2554_070318_FS_SV,2018-03-17,NaN
3,6.553432,9.958,Diethyl phthalate,96.880089,C12H14O4,84-66-2,2554_070318_FS_SV,2018-03-17,NaN
4,13.224109,NaN,Octadecanoic acid,96.826412,C18H36O2,57-11-4,2554_070318_FS_SV,2018-03-17,NaN
...,...,...,...,...,...,...,...,...,...
92895,11.813947,NaN,Allyl p-(2-hydroxyethoxy)benzoate,70.356172,C12H14O4,1000241-83-5,GW38_13_10_2020_SCAN,2020-11-17,NaN
92896,14.399062,NaN,3-Ethyl-3-methylheptane,70.297500,C10H22,17302-01-1,GW38_13_10_2020_SCAN,2020-11-17,NaN
92897,15.874923,NaN,"Thiourea, 1-benzoyl-3-(3-methyl-5-isoxazolyl)-",70.122934,C12H11N3O2S,118385-15-2,GW38_13_10_2020_SCAN,2020-11-17,NaN
92898,14.682249,NaN,"Tricyclo[3.3.1.1(3,7)]decanone, 4-iodo-, (1.al...",70.103971,C10H13IO,56781-85-2,GW38_13_10_2020_SCAN,2020-11-17,NaN


# 2. Análisis de datos anterior a la inserción

Aqui sacaremos información y conclusiones sobre los datos antes del siguiente paso: la creación de la base de datos y su estructura de tablas.

In [5]:
df_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92900 entries, 0 to 92899
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Component RT   92900 non-null  float64
 1   Library RT     3193 non-null   float64
 2   Compound Name  92900 non-null  object 
 3   Match Factor   92900 non-null  float64
 4   Formula        92900 non-null  object 
 5   CAS#           92900 non-null  object 
 6   Sample Name    92900 non-null  object 
 7   Date           92900 non-null  object 
 8   Group          0 non-null      object 
dtypes: float64(3), object(6)
memory usage: 6.4+ MB


Tenemos un total de 8 columnas o atributos, los unicos que tienen valores perdidos son Group y Library RT. 
Esto nos indica que las tablas que creemos deberan permitir NULL en ambos atributos.

In [6]:
df_total.describe()

,Component RT,Library RT,Match Factor
count,92900.000000,3193.000000,92900.000000
mean,18.452575,19.798469,79.206513
std,8.728189,8.382373,6.771769
min,3.090338,3.393000,70.000301
25%,10.757840,13.407975,73.619809
50%,17.119108,18.415000,77.914775
75%,25.278178,27.724000,83.545323
max,41.817038,36.630000,99.368738


Viendo la serie de Match Factor podemos concluir que se trata de un porcentaje ya que todos sus valores están entre 70.00 y 98.9, tendremos que añadir una restricción a la tabla donde se almacene este atributo para asegurarnos nos permitir valores negativos o superiores a 100.00.

In [7]:
df_total.duplicated().sum()

0

No tenemos tuplas repetidas, esto es importante ya que en una base de datos relacional no debemos tener redundancia o incosistencias en las tuplas.

# 3. Creación de Base de Datos (SQLite) con SQLAlchemy  

Crearemos una base de datos **SQLite** en el `database_path` indicado.  
Si ya existe una base de datos en ese directorio, será eliminada y creada de nuevo.  

In [6]:
from sqlalchemy import create_engine
from sqlalchemy_utils import database_exists, create_database

database_path = "../Database/Petrola.db"
database_url = "sqlite:///../Database/Petrola.db"

if os.path.exists(database_path):
    os.remove(database_path)
    print("Database eliminated")

# Crear la base de datos de nuevo
engine = create_engine(database_url)
if not database_exists(engine.url):
    create_database(engine.url)
print(f"Database successfully created at {database_path}")

Database successfully created at ../Database/Petrola.db


# 4. Creación de las Tablas

Con el apartado de análisis de datos, hemos llegado a las siguientes conclusiones:  

- No hay filas duplicadas, no tenemos que preocuparnos a la hora de la insercción.  
- **`Library RT`** y **`Group`** deben admitir valores perdidos (Nulos) 
- **`Match Factor`** representa un porcentaje y debe tratarse como tal.  

Para crear las tablas usamos **SQLAlchemy ORM**, de esta forma es más visual y sencillo.  
Hemos decidido dividir los atributos en tres tablas diferentes:  

### **Tabla Compounds**  
- `cas` (Primary Key)  
- `name`  
- `formula`  
- `group` (nullable)  

### **Tabla Stations**  
- `station_id` (Primary Key)
- `st_type`
- `geology`
- `well_depth`
- `elevation`
- `x` 
- `y`

### **Tabla Samples**  
- `id` (Primary Key generada automáticamente)  
- `station_id` (Foreign Key a `Stations.station_id`)  
- `compound_cas` (Foreign Key a `Compounds.cas`)  
- `component_rt`  
- `library_rt` (nullable)  
- `match_factor` (con restricción entre 0 y 100)  
- `sample_date` (tipo Date)  

Consultar la imagen `Entity Relationship Diagram.png` de la carpeta `/Database` para visualizar la estructura de las tablas.

In [7]:
from datetime import date
from sqlalchemy import ForeignKey, String, Date, Numeric, Integer, CheckConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker

Session = sessionmaker(bind=engine)
session = Session()

class Base(DeclarativeBase):
    pass

class Stations(Base):
    __tablename__ = "stations"
    
    station_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    st_type: Mapped[int] = mapped_column(Integer, nullable=False)
    geology: Mapped[str] = mapped_column(String(20), nullable=True)
    well_depth: Mapped[float] = mapped_column(Numeric(8,3), nullable=True)
    elevation: Mapped[float] = mapped_column(Numeric(8,3), nullable=True)
    x: Mapped[float] = mapped_column(Numeric(10,3), nullable=False)
    y: Mapped[float] = mapped_column(Numeric(10,3), nullable=False)
    
class Compounds(Base):
    __tablename__ = "compounds"
    
    cas: Mapped[str] = mapped_column(String(20), primary_key=True)
    name: Mapped[str] = mapped_column(String(80), nullable=False)
    formula: Mapped[str] = mapped_column(String(20), nullable=False)
    group: Mapped[str] = mapped_column(String(30), nullable=True)

class Samples(Base):
    __tablename__ = "samples"
    
    id: Mapped[int] = mapped_column(Integer, autoincrement=True, primary_key=True)
    station_id: Mapped[str] = mapped_column(ForeignKey("stations.station_id"), nullable=False)
    compound_cas: Mapped[str] = mapped_column(ForeignKey("compounds.cas"), nullable=False)
    component_rt: Mapped[float] = mapped_column(Numeric(17,13), nullable=False) #13 digitos en la parte decimal -> (0000.0000000000000 - 9999.9999999999999)
    library_rt: Mapped[float] = mapped_column(Numeric(17,13), nullable=True)
    match_factor: Mapped[float] = mapped_column(Numeric(16, 13), nullable=False) # como es un percentaje la parte entera admite solo 3 digitos y la decimal 13 digitos
    sample_date: Mapped[date] = mapped_column(Date, nullable=False)

    __table_args__ = (
        # No podemos insertar un porcentaje negativo o mayor a 100%
        CheckConstraint("match_factor >= 0 AND match_factor <= 100", name="check_match_factor_range"),
    )

Base.metadata.create_all(engine)
print("Database Tables created")

Database Tables created


In [8]:
from sqlalchemy import inspect

# Consultamos para saber si las tablas se han creado correctamente
inspector = inspect(engine)
tables = inspector.get_table_names()
for table in tables:
    print(f"Tabla: {table}")
    columns = inspector.get_columns(table)
    for column in columns:
        print(f"    Columna: {column['name']} - Tipo: {column['type']}")

Tabla: compounds
    Columna: cas - Tipo: VARCHAR(20)
    Columna: name - Tipo: VARCHAR(80)
    Columna: formula - Tipo: VARCHAR(20)
    Columna: group - Tipo: VARCHAR(30)
Tabla: samples
    Columna: id - Tipo: INTEGER
    Columna: station_id - Tipo: INTEGER
    Columna: compound_cas - Tipo: VARCHAR(20)
    Columna: component_rt - Tipo: NUMERIC(17, 13)
    Columna: library_rt - Tipo: NUMERIC(17, 13)
    Columna: match_factor - Tipo: NUMERIC(16, 13)
    Columna: sample_date - Tipo: DATE
Tabla: stations
    Columna: station_id - Tipo: INTEGER
    Columna: st_type - Tipo: INTEGER
    Columna: geology - Tipo: VARCHAR(20)
    Columna: well_depth - Tipo: NUMERIC(8, 3)
    Columna: elevation - Tipo: NUMERIC(8, 3)
    Columna: x - Tipo: NUMERIC(10, 3)
    Columna: y - Tipo: NUMERIC(10, 3)


# 5. Inserción de Tuplas  (Stations)

Primero tenemos que insertar las tuplas de la tabla Stations, por lo que recorremos el excel de la estaciones y las rellenamos una una.  

In [9]:
# Ruta del Excel
excel_path_input = "../Datos Excel/Estaciones/General.xlsx"

# Cargamos el primer excel
workbook = openpyxl.load_workbook(excel_path_input)
station_tuples = 0
i = 0
total_rows = sum(sheet.max_row - 1 for sheet in workbook.worksheets)

# Creamos las columnas Date y Group
for sheet in workbook.worksheets:
    #formated_date = datetime.strptime(normalize_dates([sheet.title])[0], "%Y-%m-%d").date()
    for row in sheet.iter_rows(min_row=2):
        i+=1
        #print(f"Row {i}/{total_rows}")
        print(f"\rRow {i}/{total_rows} - {((i/total_rows)*100):.2f}%", end="")  # Sobrescribe la línea con el nuevo progreso
        # row[0] = station_id | row[1] = st_type	| row[2] = geology	| row[3] = well_depth	
        # row[4] = elevation	    | row[5] = x	        | row[6] = y
        station = Stations(
            station_id=row[0].value,
            st_type=row[1].value,
            geology=row[2].value,
            well_depth=row[3].value,
            elevation=row[4].value,
            x=row[5].value,
            y=row[6].value
        )
        # Verificar si el componente ya existe
        existing_component = session.query(Stations).filter_by(station_id=row[0].value).first()
        if not existing_component:
            session.add(station)
            station_tuples+=1
            
        # Guardar los cambios en la base de datos
        session.commit()
print("\nAll tuples were successfully inserted into the database.")
print(f"{station_tuples} tuples inserted into Stations")

Row 100/100 - 100.00%
All tuples were successfully inserted into the database.
100 tuples inserted into Stations


In [10]:
# Comprobamos que la stations se han insertado bien
from sqlalchemy.orm import sessionmaker
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../Database/Petrola.db")

Session = sessionmaker(bind=engine)
session = Session()

all_stations = session.query(Stations).all()

for st in all_stations:
    print(f"{st.station_id} - {st.st_type} - {st.geology} - {st.well_depth} - {st.elevation} - {st.x} - {st.y}")

493 - 1 - JURÁSICO - 400.000 - 910.000 - 628525.000 - 4302550.000
699 - 1 - JURÁSICO - 200.000 - 844.140 - 619500.000 - 4300650.000
2429 - 1 - JURÁSICO - 215.000 - 891.000 - 631340.000 - 4300490.000
2430 - 1 - JURÁSICO - 108.360 - 850.000 - 619644.000 - 4300550.000
2534 - 1 - CRETÁCICO - 30.000 - 900.000 - 622443.000 - 4300541.000
2535 - 1 - CRETÁCICO - 16.000 - 873.400 - 626300.000 - 4301891.000
2536 - 4 - CRETÁCICO - 22.000 - 876.400 - 626734.000 - 4301563.000
2537 - 1 - CRETÁCICO - 7.200 - 902.000 - 625472.000 - 4301901.000
2538 - 1 - CRETÁCICO - 5.900 - 861.000 - 621632.000 - 4299836.000
2539 - 1 - CRETÁCICO - 3.300 - 861.000 - 621437.000 - 4299801.000
2540 - 1 - CRETÁCICO - 4.000 - 884.000 - 624879.000 - 4301173.000
2541 - 1 - CRETÁCICO - 40.000 - 837.000 - 620857.000 - 4302948.000
2542 - 1 - CRETÁCICO - 90.000 - 837.000 - 620806.000 - 4302927.000
2543 - 1 - JURÁSICO - 200.000 - 875.000 - 624630.000 - 4298639.000
2544 - 1 - CRETÁCICO - 20.000 - 849.200 - 625135.000 - 4299722.000
2

# Detección de filas vacias y limpieza de las mismas 
## (obsoleto: se detectan e ignoran las filas completamente vacias y evitamos su insercción. Asi nos ahorramos la limpieza de cada excel)

Openpyxl detecta mas filas con datos de las reales, por lo tanto las limpiamos. Para cada página de los excels, empezamos por la última fila con datos segun openpyxl y se es completamente vacía, la eliminamos y comprobamos la anterior, hasta encontrar una fila con datos. Esa fila será la última fila real del excel.

In [5]:
# Ruta del Excel
excels_path_input = "../Datos Excel/LecturasIncluidas"

# Contadores individuales para cada columna
component_rt_none_counter = 0
library_rt_none_counter = 0
compound_name_none_counter = 0
match_factor_none_counter = 0
formula_none_counter = 0
cas_none_counter = 0
sample_name_none_counter = 0

for excel_name in os.listdir(excels_path_input):
    
    excel_path_input = os.path.join(excels_path_input, excel_name)

    if not excel_name.endswith((".xlsx", ".xlsm")):
        continue  # Saltar archivos que no son Excel

    workbook = openpyxl.load_workbook(excel_path_input, data_only=True)

    for sheet in workbook.worksheets:
        for row in sheet.iter_rows(min_row=5):
            if row[0].value is None:
                component_rt_none_counter += 1
            if row[1].value is None:
                library_rt_none_counter += 1
            if row[2].value is None:
                compound_name_none_counter += 1
            if row[3].value is None:
                match_factor_none_counter += 1
            if row[4].value is None:
                formula_none_counter += 1
            if row[5].value is None:
                cas_none_counter += 1
            if row[6].value is None:
                sample_name_none_counter += 1

# Imprimir los resultados
print("\nResumen de valores None encontrados:")
print(f"Component RT: {component_rt_none_counter}")
print(f"Library RT: {library_rt_none_counter}")
print(f"Compound Name: {compound_name_none_counter}")
print(f"Match Factor: {match_factor_none_counter}")
print(f"Formula: {formula_none_counter}")
print(f"CAS: {cas_none_counter}")
print(f"Sample Name: {sample_name_none_counter}")

# Total de Nones
total_nones = (
    component_rt_none_counter + library_rt_none_counter + compound_name_none_counter +
    match_factor_none_counter + formula_none_counter + cas_none_counter + sample_name_none_counter
)
print(f"\nNúmero total de Nones: {total_nones}")


Resumen de valores None encontrados:
Component RT: 751
Library RT: 87429
Compound Name: 751
Match Factor: 751
Formula: 751
CAS: 751
Sample Name: 751

Número total de Nones: 91935


In [ ]:
'''# Comparación con pandas.info()
df_total.info()'''

Vemos que se han detectado muchos Nones en los campos, ademas es el mismo numero de Nones en la mayoria. Sin embargo, en el pandas.info solo tenemos valores perdidos en group y library rt. Esto muestra que hay un problema en como lee las filas openpyxl. Pandas parece no tener este problema y detectar las filas de los excels bien a la primera

In [15]:
# Ruta del Excel
'''excels_path_input = "../Datos Excel/LecturasIncluidas"

# Lista para almacenar filas completamente vacías
empty_rows = []

for excel_name in os.listdir(excels_path_input):
    
    excel_path_input = os.path.join(excels_path_input, excel_name)

    if not excel_name.endswith((".xlsx", ".xlsm")):
        continue  # Saltar archivos que no son Excel

    workbook = openpyxl.load_workbook(excel_path_input, data_only=True)

    for sheet in workbook.worksheets:
        for row in sheet.iter_rows(min_row=5, max_row=sheet.max_row):
            row_number = row[0].row  # Obtener número de fila

            # Verificar si TODAS las columnas de la fila son None
            if all(cell.value is None for cell in row[:7]):  # Solo chequeamos las primeras 7 columnas
                empty_rows.append((excel_name, sheet.title, row_number))

# Imprimir resultados
if empty_rows:
    print("\nFilas completamente vacías encontradas:")
    for excel_file, sheet_name, row_num in empty_rows:
        print(f"Archivo: {excel_file} | Hoja: {sheet_name} | Fila: {row_num}")
else:
    print("\nNo se encontraron filas completamente vacías.")'''


Filas completamente vacías encontradas:
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 271
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 272
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 273
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 274
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 275
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 276
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 277
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 278
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 279
Archivo: 2571_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023 .xlsx | Hoja: 17032018 | Fila: 280
Archivo: 2571_RESULTADOS CUAL

Si se están cargando filas completamente vacías es porque openpyxl detecta mas filas de las que realmente tienen datos en esa pagina. Es decir, 200 filas con datos pero openpyxl detecta 216, por lo que añade 16 filas con todo None, generando problemas.

In [16]:
# ACTUALIZAMOS EL VALOR DE LAST_ROW PARA ASEGURARNOS DE QUE OPENPYXL SABE CON CERTEZA CUAL ES LA ULTIMA FILA CON DATOS DE CADA PAGINA DE CADA EXCEL DE FORMA CORRECTA.

# Ruta del Excel
'''excels_path_input = "../Datos Excel/LecturasIncluidas"

for excel_name in os.listdir(excels_path_input):

    excel_path_input = os.path.join(excels_path_input, excel_name)

    # Abrimos el archivo Excel
    workbook = openpyxl.load_workbook(excel_path_input)

    for sheet in workbook.worksheets:
        last_row = sheet.max_row  # Última fila con datos según Excel
        
        # Eliminar filas vacías desde la última fila hacia atrás
        for row in range(last_row, 0, -1):  # Iterar desde la última fila hacia arriba
            # Comprobar si toda la fila está vacía
            if all(sheet.cell(row=row, column=col).value is None for col in range(1, sheet.max_column + 1)):
                sheet.delete_rows(row)  # Eliminar la fila vacía
            else:
                break  # Si encontramos una fila con datos, detenemos el bucle

        # Después de limpiar, obtenemos la última fila con datos
        last_row_cleaned = sheet.max_row
        print(f"Archivo: {excel_name} | Hoja: {sheet.title} | Última fila con datos (limpia): {last_row_cleaned}")
    
    # Guardamos los cambios en el archivo limpio
    workbook.save(excel_path_input)

print("\nLimpieza completada en todos los archivos.")'''

Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 17032018 | Última fila con datos (limpia): 325
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 16042018 | Última fila con datos (limpia): 354
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 15052018 | Última fila con datos (limpia): 350
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 18062018 | Última fila con datos (limpia): 386
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 25072018 | Última fila con datos (limpia): 429
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 31082018 | Última fila con datos (limpia): 431
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 26092018 | Última fila con datos (limpia): 430
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_2023.xlsx | Hoja: 25102018 | Última fila con datos (limpia): 312
Archivo: 2554_RESULTADOS CUALITATIVOS PE╠üTROLA 2018_202

In [18]:
# Ruta del Excel
'''excels_path_input = "../Datos Excel/LecturasIncluidas"

# Lista para almacenar filas completamente vacías
empty_rows = []

for excel_name in os.listdir(excels_path_input):
    
    excel_path_input = os.path.join(excels_path_input, excel_name)

    if not excel_name.endswith((".xlsx", ".xlsm")):
        continue  # Saltar archivos que no son Excel

    workbook = openpyxl.load_workbook(excel_path_input, data_only=True)

    for sheet in workbook.worksheets:
        for row in sheet.iter_rows(min_row=5, max_row=sheet.max_row):
            row_number = row[0].row  # Obtener número de fila

            # Verificar si TODAS las columnas de la fila son None
            if all(cell.value is None for cell in row[:7]):  # Solo chequeamos las primeras 7 columnas
                empty_rows.append((excel_name, sheet.title, row_number))

# Imprimir resultados
if empty_rows:
    print("\nFilas completamente vacías encontradas:")
    for excel_file, sheet_name, row_num in empty_rows:
        print(f"Archivo: {excel_file} | Hoja: {sheet_name} | Fila: {row_num}")
else:
    print("\nNo se encontraron filas completamente vacías.")'''


No se encontraron filas completamente vacías.


Tras la limpieza ya no tenemos el problema de filas completamente vacías, por lo que ahora si podemos proceder a la insercción de las tuplas en la base de datos.

# 6. Insercción de tuplas (Samples y Compound)

In [11]:
# Funciones auxiliares para la insercción

def sheet_is_valid(sheet):
    return not len(sheet.title) < 8

def row_is_valid(row):
    for i in range(7):
        if row[i].value not in (None, "", ''):
            return True
    return False

def excel_is_valid(excel):
    return not excel[0:2] == "GW"

In [12]:
# Ruta del Excel
excels_path_input = "../Datos Excel/LecturasIncluidas"
excel_number = 0
compound_tuples = 0
sample_tuples = 0

for excel_name in os.listdir(excels_path_input):
    
    # Si el nombre del excel no es valido, saltamos al siguiente
    if not excel_is_valid(excel_name):
        excel_number += 1
        continue
    
    # Obtenemos la ubicación del excel
    excel_path_input = os.path.join(excels_path_input, excel_name)
    excel_number += 1
    print(f"\nLoading Excel {excel_number} of {len(os.listdir(excels_path_input))}...")
    
    # Cargamos el primer excel
    workbook = openpyxl.load_workbook(excel_path_input)
    
    compound_tuples_aux = 0
    sample_tuples_aux = 0
    i = 0
    total_rows = sum(sheet.max_row - 4 for sheet in workbook.worksheets)
    
    # Creamos las columnas Date y Group
    for sheet in workbook.worksheets:
        # Si la hoja no es valida, pasamos a la siguiente
        if not sheet_is_valid(sheet):
            continue
        
        if sheet.title == "020142019": # para manejar el caso de la página con nombre en formato incorrecto y evitar el error en la función normalize_dates
            formated_date = datetime.strptime(normalize_dates(["02042019"])[0], "%Y-%m-%d").date()
        else:
            formated_date = datetime.strptime(normalize_dates([sheet.title])[0], "%Y-%m-%d").date()
        
        for row in sheet.iter_rows(min_row=5, max_row=sheet.max_row):
            i+=1
            #print(f"Row {i}/{total_rows}")
            print(f"\r\tRow {i}/{total_rows} - {((i/total_rows)*100):.2f}%", end="")  # Sobrescribe la línea con el nuevo progreso
            # row[0] = Component RT | row[1] = Library RT 	| row[2] = Compound Name	| row[3] = Match Factor	
            # row[4] = Formula	    | row[5] = CAS	        | row[6] = Sample Name
            
            # Antes de realizar la carga y la insercción comprobamos que la fila tiene datos
            if not row_is_valid(row):
                continue
            
            compound = Compounds(
                cas=row[5].value, 
                name=row[2].value, 
                formula=row[4].value, 
                group=None) # por el momento lo dejamos en None / NULL
            
            library_rt_value = row[1].value if row[1].value != '' else None
            sample = Samples(                
                station_id=row[6].value.split("_")[0], 
                compound_cas=row[5].value, 
                component_rt=row[0].value, 
                library_rt=library_rt_value,
                match_factor=row[3].value,
                sample_date=formated_date)
            
            # Verificar si el compuesto ya existe
            existing_compound = session.query(Compounds).filter_by(cas=row[5].value).first()
            if not existing_compound:
                session.add(compound)
                compound_tuples_aux+=1
            
            # Verificar si la muestra ya existe
            existing_sample = session.query(Samples).filter_by(
                 station_id=row[6].value.split("_")[0], 
                compound_cas=compound.cas,
                component_rt=row[0].value,
                library_rt=library_rt_value, #row[1].value
                match_factor=row[3].value,
                sample_date=formated_date
            ).first()
            
            if not existing_sample:
                session.add(sample)
                sample_tuples_aux+=1
                
            # Guardar los cambios en la base de datos
            session.commit()
    
    compound_tuples += compound_tuples_aux
    sample_tuples += sample_tuples_aux


         
print("\nAll tuples were successfully inserted into the database.")
print(f"{compound_tuples} tuples inserted into Compounds")
print(f"{sample_tuples} tuples inserted into Samples")


Loading Excel 1 of 19...
	Row 7010/7010 - 100.00%
Loading Excel 2 of 19...
	Row 7057/7057 - 100.00%
Loading Excel 3 of 19...
	Row 6854/6854 - 100.00%
Loading Excel 4 of 19...
	Row 459/474 - 96.84%
Loading Excel 5 of 19...
	Row 5849/5850 - 99.98%
Loading Excel 6 of 19...
	Row 4464/4464 - 100.00%
Loading Excel 7 of 19...
	Row 6809/6809 - 100.00%
Loading Excel 8 of 19...
	Row 6978/6978 - 100.00%
Loading Excel 9 of 19...
	Row 6523/6523 - 100.00%
Loading Excel 10 of 19...
	Row 5708/5708 - 100.00%
Loading Excel 11 of 19...
	Row 3859/3859 - 100.00%
Loading Excel 12 of 19...
	Row 5728/5728 - 100.00%
Loading Excel 13 of 19...
	Row 5694/5694 - 100.00%
Loading Excel 14 of 19...
	Row 2744/2744 - 100.00%
Loading Excel 15 of 19...
	Row 5542/5542 - 100.00%
Loading Excel 16 of 19...
	Row 4068/4068 - 100.00%
All tuples were successfully inserted into the database.
8903 tuples inserted into Compounds
84596 tuples inserted into Samples


In [20]:
# Consultamos las primeras 10 tuplas de la tabla 'samples' para confirmar que los datos se han insertado correctamente

from sqlalchemy.orm import sessionmaker
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../Database/Petrola.db")

Session = sessionmaker(bind=engine)
session = Session()

samples = session.query(Samples).filter(Samples.id < 11)

for sample in samples:
    print(f"{sample.id} - {sample.station_id} - {sample.compound_cas} - {sample.match_factor} - {sample.sample_date}")

1 - 2554 - 630-02-4 - 97.3448758643285 - 2018-03-17
2 - 2554 - 3622-84-2 - 97.3426256374860 - 2018-03-17
3 - 2554 - 143-07-7 - 97.0340881226838 - 2018-03-17
4 - 2554 - 84-66-2 - 96.8800885288180 - 2018-03-17
5 - 2554 - 57-11-4 - 96.8264123682283 - 2018-03-17
6 - 2554 - 629-76-5 - 96.7604533453609 - 2018-03-17
7 - 2554 - 84-74-2 - 96.7198925332146 - 2018-03-17
8 - 2554 - 111-02-4 - 96.3098274851958 - 2018-03-17
9 - 2554 - 112-92-5 - 96.1331181349050 - 2018-03-17
10 - 2554 - 119-61-9 - 96.0722380246357 - 2018-03-17
